<a href="https://colab.research.google.com/github/devesssi/llm-diffusion-models-finetuning/blob/main/finetunn(unsloth).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Wed Jan  7 08:34:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q unsloth
# Also get the latest nightly Unsloth!
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.1/381.1 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! Unsloth also supports RoPE (Rotary Positinal Embedding) scaling internally.
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, # Will load the 4Bit Quantized Model
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, # a higher alpha value assigns more weight to the LoRA activations
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.1.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from datasets import load_dataset
dataset = load_dataset("facebook/research-plan-gen", "ml", split = "train")

ml/train/data.parquet:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

ml/test/data.parquet:   0%|          | 0.00/1.85M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/685 [00:00<?, ? examples/s]

In [ ]:
print(dataset[:5])

{'Goal': ["You are tasked with fine-tuning a Large Multimodal Model (LMM) for a specific downstream task. The LMM has been pre-trained on a large corpus of data and has shown impressive capabilities across various multimodal tasks. However, full fine-tuning of the LMM is highly parameter-intensive and computationally expensive. You need to develop an efficient and effective strategy for adapting the LMM to the target task while minimizing the number of tunable parameters. The approach should also provide intuitive control over the model's behavior.", 'You are a researcher trying to train a neural network to approximate the solution to a Bayesian decision-making problem in a sensorimotor task. The task involves continuous actions, and the cost function is complex and difficult to optimize. Your goal is to develop a training method that does not require numerical solutions to the Bayesian decision-making problem. You have a dataset of parameters and sensory inputs, but no labeled data wi

In [ ]:
r1_prompt = """You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.
<problem>
{}
</problem>

{}

{}
"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    goals     = examples["Goal"]
    solutions = examples["Reference solution"]
    texts     = []

    for goal, solution in zip(goals, solutions):
        # optionally you could insert a placeholder for 'thoughts' here if you generate them later
        inner_thoughts = ""  # <-- no existing field for thoughts in this dataset
        text = r1_prompt.format(goal, inner_thoughts, solution) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)


Map:   0%|          | 0/6872 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, # Number of processors to use for processing the dataset
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2, # The batch size per GPU/TPU core
        gradient_accumulation_steps = 4, # Number of steps to perform befor each gradient accumulation
        warmup_steps = 5, # Few updates with low learning rate before actual training
        max_steps = 60, # Specifies the total number of training steps (batches) to run.
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc for observability
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/6872 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,872 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.001600
2,1.663600
3,1.848800
4,1.771700
5,1.612800
6,1.805900
7,1.458900
8,1.665900
9,1.650700
10,1.557800


In [ ]:
from unsloth.chat_templates import get_chat_template
sys_prompt = """You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.
<problem>
{}
</problem>
"""
message = sys_prompt.format("How do I handle hallucinations or incoherent outputs from my fine-tuned model ?")
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": message},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 1024, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
response = tokenizer.batch_decode(outputs)

In [ ]:
print(response[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.
<problem>
How do I handle hallucinations or incoherent outputs from my fine-tuned model?
</problem>
<|eot_id|><|start_header_id|>assistant<|end_header_id|>

To address hallucinations or incoherent outputs from my fine-tuned model, I will implement the following plan:

1. **Regularization Techniques**: I will incorporate regularization techniques such as dropout, L1/L2 regularization, or gradient clipping to limit the impact of hallucinations. These methods are designed to reduce overconfidence in model predictions and encourage the model to provide more coherent outputs.
2. **Robustn

In [ ]:
model.save_pretrained("bro-001-3B")  # Local saving
tokenizer.save_pretrained("bro-001-3B")

('bro-001-3B/tokenizer_config.json',
 'bro-001-3B/special_tokens_map.json',
 'bro-001-3B/chat_template.jinja',
 'bro-001-3B/tokenizer.json')

In [ ]:
!zip -r bro-001-3B.zip bro-001-3B

  adding: bro-001-3B/ (stored 0%)
  adding: bro-001-3B/tokenizer_config.json (deflated 96%)
  adding: bro-001-3B/tokenizer.json (deflated 85%)
  adding: bro-001-3B/adapter_config.json (deflated 58%)
  adding: bro-001-3B/chat_template.jinja (deflated 72%)
  adding: bro-001-3B/special_tokens_map.json (deflated 71%)
  adding: bro-001-3B/README.md (deflated 65%)
  adding: bro-001-3B/adapter_model.safetensors (deflated 7%)


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
FastLanguageModel.for_inference(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNo

In [ ]:
prompt = "Task:\nDesign a research plan to reduce hallucinations in large language models.\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=600,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


Task:
Design a research plan to reduce hallucinations in large language models.
Hallucinations in large language models occur when the model generates text that is not grounded in reality. This can be caused by various factors such as incomplete training data, overfitting, or the model's ability to generate coherent text. To reduce hallucinations, we need to design a research plan that focuses on understanding the causes of hallucinations, developing new evaluation metrics, and designing novel architectures that can better handle ungrounded text.

Research Questions:
1. What are the primary causes of hallucinations in large language models?
2. How can we develop effective evaluation metrics to detect hallucinations?
3. Can we design novel architectures that can better handle ungrounded text?

Objectives:
1. Investigate the relationship between hallucinations and model architecture.
2. Develop a new evaluation metric that can detect hallucinations.
3. Design and test novel architectures

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./bro-001-3B",
    max_seq_length=2048,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
prompt = """Task:
Design a research plan to reduce hallucinations in large language models.
"""

# -----------------------------
# 4. TOKENIZE + MOVE TO GPU
# -----------------------------
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# -----------------------------
# 5. GENERATE
# -----------------------------
output = model.generate(
    **inputs,
    max_new_tokens=1200,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

# -----------------------------
# 6. DECODE OUTPUT
# -----------------------------
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

In [ ]:
from google.colab import files
files.upload()


In [ ]:
!unzip bro-001-3B.zip

Archive:  bro-001-3B.zip
   creating: bro-001-3B/
  inflating: bro-001-3B/tokenizer_config.json  
  inflating: bro-001-3B/tokenizer.json  
  inflating: bro-001-3B/adapter_config.json  
  inflating: bro-001-3B/chat_template.jinja  
  inflating: bro-001-3B/special_tokens_map.json  
  inflating: bro-001-3B/README.md    
  inflating: bro-001-3B/adapter_model.safetensors  


In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.8Gi       9.7Gi       3.0Mi       1.2Gi        10Gi
Swap:             0B          0B          0B


In [ ]:
from huggingface_hub import login
login()


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model_id = "unsloth/Llama-3.2-3B-Instruct"
lora_path = "/content/bro-001-3B"
output_path = "/content/bro-001-3B-merged1"

print("Loading base model on GPU...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, lora_path)

print("Merging LoRA...")
model = model.merge_and_unload()

print("Moving model to CPU...")
model = model.to("cpu")

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("Saving merged model (bin format, faster)...")
model.save_pretrained(output_path, safe_serialization=False)
tokenizer.save_pretrained(output_path)

print("✅ Merge + save completed")


Loading base model on GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading LoRA adapter...
Merging LoRA...
Moving model to CPU...
Saving merged model (bin format, faster)...
✅ Merge + save completed


In [ ]:
%cd /content
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!pip install -r requirements.txt


/content
fatal: destination path 'llama.cpp' already exists and is not an empty directory.
/content/llama.cpp
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment


In [ ]:
!python convert_hf_to_gguf.py \
  /content/bro-001-3B-merged1 \
  --outfile /content/bro-001-3B.gguf


INFO:hf-to-gguf:Loading model: bro-001-3B-merged1
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'pytorch_model.bin.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'pytorch_model-00001-of-00002.bin'
INFO:hf-to-gguf:gguf: indexing model part 'pytorch_model-00002-of-00002.bin'
INFO:hf-to-gguf:heuristics detected float16 tensor dtype, setting --outtype f16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_q.weight,         torch.float16 --> F16, shape = {3072, 3072}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.attn_v.weight,         torch.float16 --> F16, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.attn_output.wei

In [ ]:
!build/bin/quantize \
  /content/bro-001-3B.gguf \
  /content/bro-001-3B-q4.gguf \
  q4_K_M


/bin/bash: line 1: build/bin/quantize: No such file or directory


In [ ]:
%cd /content/llama.cpp


/content/llama.cpp


In [ ]:
!ls

AGENTS.md		       convert_lora_to_gguf.py	pocs
AUTHORS			       docs			poetry.lock
benches			       examples			pyproject.toml
build-xcframework.sh	       flake.lock		pyrightconfig.json
ci			       flake.nix		README.md
CLAUDE.md		       ggml			requirements
cmake			       gguf-py			requirements.txt
CMakeLists.txt		       grammars			scripts
CMakePresets.json	       include			SECURITY.md
CODEOWNERS		       LICENSE			src
common			       licenses			tests
CONTRIBUTING.md		       Makefile			tools
convert_hf_to_gguf.py	       media			vendor
convert_hf_to_gguf_update.py   models
convert_llama_ggml_to_gguf.py  mypy.ini


In [ ]:
!cmake -B build


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [ ]:
!cmake --build build --config Release

[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  0%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-threading.cpp.o
[  1%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-quants.c.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/gguf.cpp.o
[  2%] Linking CXX shared library ../../bin/libggml-base.so
[  2%] Built target ggml-base
[  2%] Building C object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/ggml-cpu.c.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/ggml-cpu.cpp.o
[  3%] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/repack.cpp.o
[  3%] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-

In [ ]:
!ls build/bin


libggml-base.so		       llama-q8dot
libggml-base.so.0	       llama-quantize
libggml-base.so.0.9.5	       llama-qwen2vl-cli
libggml-cpu.so		       llama-retrieval
libggml-cpu.so.0	       llama-save-load-state
libggml-cpu.so.0.9.5	       llama-server
libggml.so		       llama-simple
libggml.so.0		       llama-simple-chat
libggml.so.0.9.5	       llama-speculative
libllama.so		       llama-speculative-simple
libllama.so.0		       llama-tokenize
libllama.so.0.0.7670	       llama-tts
libmtmd.so		       llama-vdot
libmtmd.so.0		       test-alloc
libmtmd.so.0.0.7670	       test-arg-parser
llama-batched		       test-autorelease
llama-batched-bench	       test-backend-ops
llama-bench		       test-backend-sampler
llama-cli		       test-barrier
llama-completion	       test-c
llama-convert-llama2c-to-ggml  test-chat
llama-cvector-generator        test-chat-parser
llama-debug		       test-chat-peg-parser
llama-diffusion-cli	       test-chat-template
llama-embedding		       test-gbnf-validator
llama-e

In [ ]:
!build/bin/llama-quantize \
  /content/bro-001-3B.gguf \
  /content/bro-001-3B-q4.gguf \
  q4_K_M


main: build = 7670 (9c142e3a2)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/bro-001-3B.gguf' to '/content/bro-001-3B-q4.gguf' as Q4_K_M
llama_model_loader: direct I/O is enabled, disabling mmap
llama_model_loader: loaded meta data with 33 key-value pairs and 256 tensors from /content/bro-001-3B.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_p f32              = 0.900000
llama_model_loader: - kv   3:                      general.sampling.temp f32              = 0.600000
llama_model_loader: - kv   4:                               general.name str              = Bro 001 3B Merged1
llama_model_loader: - kv   5:         

In [ ]:
!ls -lh /content/bro-001-3B-q4.gguf


-rw-r--r-- 1 root root 2.1G Jan  8 09:11 /content/bro-001-3B-q4.gguf


In [ ]:
from google.colab import files
files.download("/content/bro-001-3B-q4.gguf")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python convert_hf_to_gguf.py \
  --model /content/bro-001-3B-merged1 \
  --outfile /content/bro-001-3B.gguf

usage: convert_hf_to_gguf.py [-h] [--vocab-only] [--outfile OUTFILE]
                             [--outtype {f32,f16,bf16,q8_0,tq1_0,tq2_0,auto}]
                             [--bigendian] [--use-temp-file] [--no-lazy]
                             [--model-name MODEL_NAME] [--verbose]
                             [--split-max-tensors SPLIT_MAX_TENSORS]
                             [--split-max-size SPLIT_MAX_SIZE] [--dry-run]
                             [--no-tensor-first-split] [--metadata METADATA]
                             [--print-supported-models] [--remote] [--mmproj]
                             [--mistral-format]
                             [--disable-mistral-community-chat-template]
                             [--sentence-transformers-dense-modules]
                             [model]
convert_hf_to_gguf.py: error: the following arguments are required: model


In [ ]:
!cmake -B build
!cmake --build build --config Release

In [ ]:
!build/bin/quantize \
  /content/bro-001-3B.gguf \
  /content/bro-001-3B-q4.gguf \
  q4_K_M

In [ ]:
from google.colab import files
files.download("/content/bro-001-3B-q4.gguf")